In [1]:
import torch
from utils.loading_utils import load_model, get_device
from data_loader.dataset import VoxelGridDataset
from matplotlib import pyplot as plt
from os.path import join, basename
import numpy as np
import json
import argparse
from utils.timers import cuda_timers
import time
import shutil
import os
from psf_depth_reconstructor import PSFDepthReconstructor
from options.inference_options import set_depth_inference_options
from data_loader.dataset import UpsampledFramesDataset
from types import SimpleNamespace
from model.unet import *
from types import SimpleNamespace
import torch
from model.model import E2VIDRecurrentPSF  
from model.model import E2VIDRecurrent
from model import unet as unet_mod          
from torch.utils.data import DataLoader
from data_loader.dataset import SequenceUpsampledFramesDataset
import matplotlib.image as mpimg
from model.unet import compute_event_frame, event_frames_to_voxel_grid
import torch.nn.functional as F

/home/yl3836/.conda/envs/mevent/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Defind Plot Grid Funcion
def plotgrid(stack):
    num_psf = stack.shape[0]  

    ncols = 5  # 원하는대로
    nrows = (num_psf + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(2*ncols, 2*nrows))

    for i in range(num_psf):
        row, col = divmod(i, ncols)
        ax = axes[row, col]
        ax.imshow(stack[i].detach().numpy(), cmap='gray')
        ax.set_title(f"Depth {i+2}")   # depth index는 2부터일 수도 있으니 맞게 조절
        ax.axis('off')

    for i in range(num_psf, nrows*ncols):
        row, col = divmod(i, ncols)
        axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
%load_ext autoreload
%autoreload 2

PATH_TO_MODEL = '/home/yl3836/mono_event/rpg_e2depth/e2depth/saved/1003_off_rotated_0_5/example_psf/model_best.pth.tar'
tag = " [rotated]"
BASE_FOLDER = '/home/yl3836/DENSE/valid_upsampled/valid_sequence_00_town06/'
#BASE_FOLDER = '/home/yl3836/DENSE/train_upsampled/train_sequence_03_town04/'
USE_GPU = True

args = SimpleNamespace(
    path_to_model=PATH_TO_MODEL,
    base_folder=BASE_FOLDER,
    use_gpu=USE_GPU,
)

In [4]:
#### Load model
PATH = args.path_to_model
ckpt = torch.load(PATH, map_location='cpu')

#### Same config as used in training
cfg = dict(num_bins=5, skip_type='sum', recurrent_block_type='convlstm',
           num_encoders=3, base_num_channels=32,
           num_residual_blocks=2, use_upsample_conv=True, norm='none', psf_init='random')



#### Top-level model class
model = E2VIDRecurrentPSF(cfg)


######## Loading the weights
model.load_state_dict(ckpt['state_dict'])  

######## Evaluation mode : disable dropout
model.eval()


#### Load optimized psf
net = model.unetrecurrentpsf

Using UpsampleConvLayer (slow, but no checkerboard artefacts)


/home/yl3836/mono_event/rpg_e2depth/e2depth/model/unet.py:51: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  th = torch.tensor(theta_deg, device=device, dtype=dtype) * torch.pi / 180
/home/yl3836/mono_event/rpg_e2depth/e2depth/model/unet.py:400: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  psf_list.append(torch.tensor(conv,dtype = torch.float32))


In [5]:
#### CPU safe collate copied from train.py 

def _to_device(x, device):
    if isinstance(x, torch.Tensor):
        return x.to(device, non_blocking=True)
    if isinstance(x, dict):
        return {k: _to_device(v, device) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return type(x)(_to_device(v, device) for v in x)
    return x

def collate_keep_sequence(batch, device):
    # batch: [N] where each item is a sequence [L] of dicts
    return _to_device(batch, device)

In [ ]:
device = 'cpu' 

unet_mod.gpu = device   # used by UNetRecurrentPSF to move tensors
model.to(device).eval() 

#### load dataset
ds = SequenceUpsampledFramesDataset(
    base_folder=BASE_FOLDER,
    event_folder='',
    depth_folder='imgs',
    frame_folder='frames',
    sequence_length=20, step_size=1, normalize=True
)

loader = DataLoader(
    ds, batch_size=1, shuffle=False,
    collate_fn=lambda b: collate_keep_sequence(b, device),
    num_workers=0, pin_memory=False
)

batch = next(iter(loader))                

##### Forming sequence, copied from lstm.py
sequence = list(map(list, zip(*batch)))   
cur_input   = sequence[0]                 
prev_states = None

#### This is the forward model of top-level model
with torch.no_grad():
    voxel, frame, pred, states, timing = model.unetrecurrentpsf(cur_input, prev_states, downsample=False, measure_time = True)


torch.Size([1, 5, 260, 346]) torch.Size([9, 1, 260, 346]) torch.Size([1, 1, 260, 346])


In [ ]:
# ==== Fast Mean + Std MSE over validation set ====
import os
import torch
import torch.nn.functional as F
import math

# --- MINIMAL CHANGE: cap CPU usage on login nodes ---
torch.set_num_threads(1)            # <<< limit intra-op threads
torch.set_num_interop_threads(1)    # <<< limit inter-op parallelism

# (Optional but harmless) also hint BLAS libs; OK even after imports
os.environ.setdefault("OMP_NUM_THREADS", "1")       # <<<
os.environ.setdefault("MKL_NUM_THREADS", "1")       # <<<
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")  # <<<
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")   # <<<

# Speed toggles
torch.backends.cudnn.benchmark = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

device = next(model.parameters()).device
use_amp = (device.type == "cuda")

# ---- OPTIONAL: keep DataLoader from spawning CPU workers ----
# If you (re)build the loader here, keep num_workers=0 to avoid extra CPU:
# from torch.utils.data import DataLoader
# loader = DataLoader(
#     ds, batch_size=1, shuffle=False,
#     num_workers=0,                      # <<< single-process loading (lowest CPU)
#     pin_memory=(device.type=="cuda"),
#     prefetch_factor=2, persistent_workers=False,
#     collate_fn=lambda b: collate_keep_sequence(b, device),
#     drop_last=False
# )

def ensure_batched(x: torch.Tensor) -> torch.Tensor:
    # Make sure there is a batch dim: [B, ...]
    return x if x.dim() >= 3 else x.unsqueeze(0)

# Running sums (stay on GPU)
mse_sum     = torch.zeros((), device=device)  # sum of per-item MSEs
mse_sq_sum  = torch.zeros((), device=device)  # sum of squares of per-item MSEs
count_items = 0

model.eval()
autocast_ctx = torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.float16) if use_amp else torch.no_grad()

with torch.inference_mode():
    with (torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.float16) if use_amp else torch.no_grad()):
        for batch in loader:
            sequence    = list(map(list, zip(*batch)))
            cur_input   = sequence[0]
            prev_states = None

            voxel, frame, pred, states = model.unetrecurrentpsf(
                cur_input, prev_states, downsample=False, measure_time=False
            )

            y_pred = pred.float().to(device, non_blocking=True).squeeze()
            y_true = frame[1].float().to(device, non_blocking=True).squeeze()

            y_pred = ensure_batched(y_pred)
            y_true = ensure_batched(y_true)

            diff = (y_pred - y_true).pow(2)
            mse_per_item = diff.flatten(1).mean(dim=1)

            mse_sum    += mse_per_item.sum()
            mse_sq_sum += (mse_per_item ** 2).sum()
            count_items += mse_per_item.numel()

count = max(count_items, 1)
mean_mse = (mse_sum / count).item()
var_pop = (mse_sq_sum / count) - (mean_mse ** 2)
std_pop = float(torch.clamp(var_pop, min=0).sqrt().item())
if count > 1:
    var_sample = (mse_sq_sum - count * (mean_mse ** 2)) / (count - 1)
    std_sample = float(torch.clamp(var_sample, min=0).sqrt().item())
else:
    std_sample = float("nan")

print(f"Dataset: {BASE_FOLDER}")
print(f"Items evaluated: {count}")
print(f"Mean MSE: {mean_mse:.6f}")
print(f"Std MSE (population): {std_pop:.6f}")
print(f"Std MSE (sample):     {std_sample:.6f}")


[W NNPACK.cpp:51] Could not initialize NNPACK! Reason: Unsupported hardware.
